## Dependency Installation

In [ ]:
# ================================
# Install Dependencies
# ================================

# LLM & LangChain
!pip install langchain
!pip install langchain_google_genai
!pip install langchain_groq
!pip install langchain_huggingface
!pip install langchain_text_splitters
!pip install langchain_community
!pip install google-generativeai

# YouTube Processing
!pip install pytube
!pip install youtube-transcript-api
!pip install yt-dlp

# Web Requests
!pip install requests
!pip install beautifulsoup4

# PDF Generation
!pip install reportlab

## Importing Required Libraries

In [ ]:
# ================================
# Standard Library
# ================================

import os        # Environment variables and system operations
import re        # Regex operations (extract video ID, clean text)
import zipfile   # Create zip files (website export)


# ================================
# Third-Party Libraries
# ================================

import requests  # Fetch subtitles / web content
import yt_dlp    # YouTube metadata & audio extraction
from pytube import YouTube  # Pytube fallback


# ================================
# Google Colab Secrets
# ================================

from google.colab import userdata  # Store API keys securely


# ================================
# YouTube Transcript Extraction
# ================================

from youtube_transcript_api import YouTubeTranscriptApi  # Direct transcript extraction


# ================================
# LLM Models
# ================================

from langchain_google_genai import ChatGoogleGenerativeAI  # Gemini model
from langchain_groq import ChatGroq                        # Groq fallback


# ================================
# Text Splitting
# ================================

from langchain_text_splitters import RecursiveCharacterTextSplitter  # Adaptive chunking


# ================================
# YouTube Loader
# ================================

from langchain_community.document_loaders import YoutubeLoader  # LangChain loader


# ================================
# Prompt Templates
# ================================

from langchain_core.prompts import (
    ChatPromptTemplate,          # Structured prompts
    SystemMessagePromptTemplate, # System instructions
    HumanMessagePromptTemplate   # User instructions
)


# ================================
# LangChain Runnables
# ================================

from langchain_core.runnables import (
    RunnablePassthrough,  # Pass input unchanged
    RunnableParallel,     # Parallel execution (future scaling)
    RunnableBranch,       # Smart routing
    RunnableLambda        # Convert functions into pipelines
)


# ================================
# Output Parser
# ================================

from langchain_core.output_parsers import StrOutputParser  # Convert LLM output to string


# ================================
# PDF Generation
# ================================

from reportlab.platypus import (
    SimpleDocTemplate,  # Create PDF
    Paragraph,          # Add formatted text
    Spacer,             # Vertical spacing
    PageBreak           # Page break
)

from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle  # Styles
from reportlab.lib.enums import TA_CENTER  # Text alignment
from reportlab.lib import colors  # Colors

## API Keys Setup

In [ ]:
# ================================
# API Keys Setup
# ================================

# Get Gemini API key from Colab secrets
gemini_api_key = userdata.get("gemini")
os.environ["GOOGLE_API_KEY"] = gemini_api_key  # Set Gemini API key


# Get Groq API key from Colab secrets
groq_api_key = userdata.get("groq")
os.environ["GROQ_API_KEY"] = groq_api_key  # Set Groq API key

## LLM Models

In [ ]:
# ================================
# Gemini Model (Primary Model)
# ================================

def get_gemini():
    """
    Returns Gemini LLM model.

    Used as the primary model for summarization.
    """

    return ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        max_tokens=8192
    )


# ================================
# Groq Model (Fallback Model)
# ================================

def get_groq():
    """
    Returns Groq LLM model.

    Used when:
    - Gemini quota exhausted
    - Gemini API error
    - Rate limits
    """

    return ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0,
        max_tokens=8000
    )

## Adaptive Chunking

In [ ]:
# ================================
# Adaptive Chunking
# ================================

def get_adaptive_chunks(text):
    """
    Splits long transcript into adaptive chunks
    for recursive summarization.
    """

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,      # Size of each chunk
        chunk_overlap=120,   # Overlap for context continuity
        separators=["\n\n", "\n", ".", " ", ""]
    )

    return splitter.split_text(text)

## Video ID Extraction & Transcript Cleaning

In [ ]:
# ================================
# Extract YouTube Video ID
# ================================

def extract_video_id(url):
    """
    Extracts video ID from a YouTube URL.

    Supports:
    - https://www.youtube.com/watch?v=xxxx
    - https://youtu.be/xxxx
    """

    pattern = r"(?:v=|\/)([0-9A-Za-z_-]{11})"  # Regex pattern

    match = re.search(pattern, url)

    if match:
        return match.group(1)

    raise ValueError("Could not extract video ID. Check URL.")


# ================================
# Clean Transcript Text
# ================================

def clean_transcript(text):
    """
    Cleans raw transcript text.

    Removes:
    - [Music], [Applause]
    - <think> reasoning blocks
    - "Okay..." reasoning text
    - Extra whitespace
    - Blank lines
    """

    # Remove [Music]
    text = re.sub(r"\[.*?\]", "", text)

    # Clean whitespace
    text = re.sub(r"\n+", "\n", text)     # Remove extra newlines
    text = re.sub(r"\s+", " ", text)      # Remove extra spaces

    return text.strip()

## Pytube Transcript Fallback

In [ ]:
# ================================
# Pytube Transcript Fallback
# ================================

def extract_pytube_transcript(url):
    """
    Extract transcript using Pytube fallback.
    Used when other transcript methods fail.
    """

    try:
        print("Using Pytube fallback")

        yt = YouTube(url)
        caption = yt.captions.get_by_language_code("en")  # English captions

        if caption:
            text = caption.generate_srt_captions()
            return clean_transcript(text)

    except Exception:
        print("Pytube failed")

    return None

## Multi-Fallback Transcript Extraction

In [ ]:
# ================================
# Extract YouTube Transcript (Multi-Fallback)
# ================================

def extract_transcript(url):
    """
    Extract transcript using multiple fallback methods.

    Fallback Order:
    1. LangChain Loader
    2. YouTube Transcript API
    3. yt-dlp subtitles
    4. Pytube captions
    """

    video_id = extract_video_id(url)


    # ================================
    # 1. LangChain Loader
    # ================================
    try:
        print("Using LangChain Loader")

        loader = YoutubeLoader.from_youtube_url(url)
        docs = loader.load()

        if docs:
            text = docs[0].page_content
            return clean_transcript(text)

    except Exception:
        print("LangChain failed")


    # ================================
    # 2. YouTube Transcript API
    # ================================
    try:
        print("Using YouTubeTranscriptApi")

        ytt_api = YouTubeTranscriptApi()

        transcript = ytt_api.fetch(
            video_id=video_id,
            languages=["en", "en-IN", "en-GB"]
        )

        text = " ".join([t["text"] for t in transcript])
        return clean_transcript(text)

    except Exception:
        print("Transcript API failed")


    # ================================
    # 3. yt-dlp Subtitle Extraction
    # ================================
    try:
        print("Using yt-dlp")

        ydl_opts = {
            "skip_download": True,
            "writesubtitles": True,
            "writeautomaticsub": True,
            "quiet": True,
            "no_warnings": True
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)

            subtitles = info.get("subtitles") or info.get("automatic_captions")

            if subtitles:

                preferred_langs = ["en", "en-IN", "en-US"]
                link = None

                for lang in preferred_langs:
                    if lang in subtitles:
                        link = subtitles[lang][0]["url"]
                        break
                else:
                    first_lang = list(subtitles.keys())[0]
                    link = subtitles[first_lang][0]["url"]

                headers = {"User-Agent": "Mozilla/5.0"}
                response = requests.get(link, headers=headers)
                response.raise_for_status()

                content_type = response.headers.get("Content-Type", "")

                if "application/json" in content_type:

                    data = response.json()

                    text = " ".join(
                        seg["utf8"]
                        for event in data.get("events", [])
                        if "segs" in event
                        for seg in event["segs"]
                    )

                else:
                    text = response.text

                return clean_transcript(text)

    except Exception:
        print("yt-dlp failed")


    # ================================
    # 4. Pytube Fallback
    # ================================
    try:
      print("Using Pytube fallback")

      text = extract_pytube_transcript(url)

      if text:
        return clean_transcript(text)

    except Exception:
        print("Pytube fallback failed")


    return None

## Audio & Visual Fallback Pipeline

In [ ]:
# ================================
# Extract Audio (Fallback Method)
# ================================

def extract_audio_transcript(url):
    """
    Extract audio when transcript is unavailable.
    Placeholder for future speech-to-text integration.
    """

    try:
        print("Extracting Audio")

        ydl_opts = {
            "format": "bestaudio/best",
            "outtmpl": "audio.%(ext)s",
            "quiet": True
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        return None

    except Exception as e:
        print("Audio extraction failed:", e)
        return None


# ================================
# Visual Frame Analysis (Fallback)
# ================================

def analyze_video_frames(url):
    """
    Handles silent videos with no transcript/audio.
    Placeholder for future vision model integration.
    """

    print("Analyzing frames (silent video fallback)")
    return "This appears to be a silent visual video."


# ================================
# Main Video Processing Pipeline
# ================================

def process_video(url):
    """
    Main pipeline:
    1. Transcript extraction
    2. Audio fallback
    3. Visual fallback
    """

    # Transcript
    transcript = extract_transcript(url)

    if transcript:
        print("Transcript found")
        return transcript


    # Audio fallback
    audio = extract_audio_transcript(url)

    if audio:
        print("Audio transcript found")
        return audio


    # Visual fallback
    return analyze_video_frames(url)

## Clean LLM Output

In [ ]:
def clean_model_output(text):
    """
    Clean model output safely without losing content
    """

    # Remove <think> blocks only
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)

    # Remove leading reasoning only if at beginning
    text = re.sub(
        r"^(Okay|Let me|I need to|First).*?\n",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove excessive blank lines (NOT spaces)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

# **1. Base Summarizer(Short Videos)**

## Base Summarizer Prompt

In [ ]:
# ================================
# Base Summarizer Prompt
# ================================

def base_summarizer_prompt(text):
    """
    Creates summarization prompt for short videos.
    Converts transcript into structured article.
    """

    system_prompt = """
You are a professional content writer.

IMPORTANT:
You are NOT summarizing.
You are converting transcript into structured article.

Your goal:
- Preserve ALL important information
- Maintain clarity and structure
- Organize content logically

STRICT RULES:
- DO NOT summarize
- DO NOT remove important details
- DO NOT compress content excessively
- Preserve examples, steps, numbers, and technical details
- Maintain original meaning

DO NOT include:
- reasoning
- thinking
- planning
- explanations
- meta commentary
- "Okay"
- "Let me"
- "I need to"

IGNORE:
- welcome messages
- subscribe requests
- promotions
- branding
- filler phrases

FORMATTING RULES:
- Use bold headings only using **Heading**
- DO NOT use markdown headings like #, ##, ###
- DO NOT use horizontal rules like --- or ***
- Keep formatting consistent


SUPPORT ALL CONTENT TYPES:
- Tutorials
- Podcasts
- Interviews
- Educational videos
- Business content
- Movie reviews
- Technical explanations
- Story-based content
- Any video content


MANDATORY STRUCTURE:

Title:
Introduction:
Main Points:
Key Takeaways:
Summary:

STYLE:
- Professional tone
- Clear structure
- Concise but complete
- Actionable where possible

Return ONLY the article.
"""

    human_prompt = """
Convert this video content into an engaging article.

Structure:
- Title
- Introduction
- Main Points
- Key Takeaways
- Summary

Here is the raw transcript:
{text}
"""

    # Create prompt template
    summarizer_prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_prompt),
        HumanMessagePromptTemplate.from_template(human_prompt)
    ])

    return summarizer_prompt

## Base Summarizer Function

In [ ]:
def get_base_summarizer(text):
    """
    Summarizes short videos using base summarizer.

    """
    try:
        print("Using Gemini")
        llm = get_gemini()

        chain = (
            RunnableLambda(base_summarizer_prompt)
            | llm
            | StrOutputParser()
        )
        response = chain.invoke(text)
        return clean_model_output(response)

    except Exception:

        print("Gemini failed → Using Groq")

        llm = get_groq()

        chain = (
            RunnableLambda(base_summarizer_prompt)
            | llm
            | StrOutputParser()
        )
        response = chain.invoke(text)
        return clean_model_output(response)

# **2. Recursive Summarizer(Long Videos)**

## Recursive Summarizer Prompt

In [ ]:
# ================================
# Recursive Summarizer Prompt
# ================================

def recursive_summarizer_prompt():
    """
    Creates recursive summarization prompt for long videos.
    """

    system_prompt = """
You are a recursive content structuring engine.

IMPORTANT:
You are NOT summarizing.
You are merging content into a structured article.

Your goal:
- Merge new content into existing article
- Preserve ALL previous information
- Add new information logically
- Avoid duplication only if identical content appears

STRICT RULES:
- DO NOT summarize
- DO NOT remove important details
- DO NOT compress content excessively
- Preserve technical details and examples

FORMATTING RULES:
- Use bold headings only using **Heading**
- DO NOT use markdown headings like #, ##, ###
- DO NOT use horizontal rules like --- or ***
- Keep formatting consistent

DO NOT include:
- reasoning
- thinking
- planning
- meta commentary

SUPPORT ALL CONTENT TYPES:
- Tutorials
- Podcasts
- Interviews
- Educational videos
- Business content
- Movie reviews
- Technical explanations
- Story-based content
- Any video content

Maintain structure:

Title:
Introduction:
Main Points:
Key Takeaways:
Summary:

STYLE:
- Professional
- Clear
- Well organized

Return ONLY updated article.
"""

    human_prompt = """
Current Summary:
{running_summary}

New Content:
{chunk}

Update the summary and maintain structure:
- Title
- Introduction
- Main Points
- Key Takeaways
- Summary
"""

    summarizer_prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_prompt),
        HumanMessagePromptTemplate.from_template(human_prompt)
    ])

    return summarizer_prompt

## Recursive Summarizer Function

In [ ]:
# ================================
# Recursive Summarization Engine
# ================================

def get_recursive_summarize(text):
    """
    Performs recursive summarization for long transcripts.
    """

    chunks = get_adaptive_chunks(text)   # Split transcript
    running_summary = ""                 # Initialize summary

    prompt = recursive_summarizer_prompt()

    for chunk in chunks:

        try:
            print("Using Gemini")

            chain = (
                prompt
                | get_gemini()
                | StrOutputParser()
            )

            updated_summary = chain.invoke({
                "running_summary": running_summary,
                "chunk": chunk
            })

        except Exception:

            print("Gemini failed → Using Groq")

            chain = (
                prompt
                | get_groq()
                | StrOutputParser()
            )

            updated_summary = chain.invoke({
                "running_summary": running_summary,
                "chunk": chunk
            })

        running_summary = clean_model_output(updated_summary)

    return running_summary

## Smart Summarizer

In [ ]:
# ================================
# Smart Summarizer
# ================================

def get_smart_summarizer(url):
    """
    Automatically selects summarization strategy
    based on transcript length.
    """

    transcript = process_video(url)

    if transcript is None:
        return "Could not extract transcript"


    # Long video
    if len(transcript.split()) >= 1000:
        print("Using recursive summarizer")
        return get_recursive_summarizer(transcript)


    # Short video
    else:
        print("Using base summarizer")
        return get_base_summarizer(transcript)

## PDF Generator

In [ ]:
# ================================
# Generate PDF
# ================================

def generate_pdf(article, filename="summary.pdf"):
    """
    Generate visually clean PDF from article
    """

    # Load default ReportLab styles
    styles = getSampleStyleSheet()

    # Define styles for different sections
    title_style = styles["Heading1"]    # Main title
    heading_style = styles["Heading2"]  # Section headings
    body_style = styles["BodyText"]     # Normal text

    story = []  # Container for PDF content

    # Add document title at top
    story.append(Paragraph("YouTube Video Summary", title_style))
    story.append(Spacer(1, 16))  # Space after title

    # Split article into lines
    lines = article.split("\n")

    # Loop through each line and format accordingly
    for line in lines:

        line = line.strip()  # Remove extra spaces

        # Skip empty lines
        if not line:
            continue

        # Format Title section
        if line.startswith("Title:"):
            story.append(Spacer(1, 10))  # Add space before section
            story.append(Paragraph(line, title_style))

        # Format Introduction section
        elif line.startswith("Introduction:"):
            story.append(Spacer(1, 10))
            story.append(Paragraph(line, heading_style))

        # Format Main Points section
        elif line.startswith("Main Points:"):
            story.append(Spacer(1, 10))
            story.append(Paragraph(line, heading_style))

        # Format Key Takeaways section
        elif line.startswith("Key Takeaways:"):
            story.append(Spacer(1, 10))
            story.append(Paragraph(line, heading_style))

        # Format Summary section
        elif line.startswith("Summary:"):
            story.append(Spacer(1, 10))
            story.append(Paragraph(line, heading_style))

        # Format normal paragraph text
        else:
            story.append(Paragraph(line, body_style))

        # Add space after each line
        story.append(Spacer(1, 6))

    # Create PDF document
    doc = SimpleDocTemplate(filename)

    # Build PDF
    doc.build(story)

    print("PDF Generated")

## Website Formatter Prompt

In [ ]:
# ================================
# Website Formatter Prompt
# ================================

def website_formatter_prompt(article):

    system_prompt = """
        You are a Senior Frontend Web Developer with 10+ years experience in HTML5, CSS3, and modern JavaScript (ES6+).

        Your task: Generate COMPLETE, PRODUCTION-READY frontend code based on user requirements.

        **MANDATORY OUTPUT FORMAT** (exact delimiters):
        --html--
        [html code here]
        --html--

        --css--
        [css code here]
        --css--

        --js--
        [java script code here]
        --js--

        DO NOT:
        - use markdown
        - write ### CSS
        - write explanations
        - write ```html
     """

    human_prompt = '''
        Create a **production-ready article webpages** in the style of **Medium, Dev.to, Hashnode, and Substack**.

        **MANDATORY REQUIREMENTS**:
        - **Mobile-first responsive design** (perfect on all devices)
        - **Clean, modern typography** (system fonts + readability first)
        - **Medium-like article layout** with card-based design
        - **Dark/light theme toggle**
        - **Smooth animations** and **scroll effects**
        - **SEO optimized** with proper meta tags
        - **Accessibility compliant** (ARIA labels, keyboard navigation)

        **CONTENT TO USE**: {article}
    '''

    return ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_prompt),
        HumanMessagePromptTemplate.from_template(human_prompt)
    ])

## Website Generator

In [ ]:
# ================================
# Generate Website
# ================================

def generate_website(article):
    """
    Generate website files (HTML, CSS, JS) from article.
    """

    try:
        print("Using Gemini")

        chain = (
            RunnableLambda(website_formatter_prompt)
            | get_gemini()
            | StrOutputParser()
        )

        website = chain.invoke({"article": article})

    except Exception:
        print("Gemini failed → Using Groq")

        chain = (
            RunnableLambda(website_formatter_prompt)
            | get_groq()
            | StrOutputParser()
        )

        website = chain.invoke({"article": article})

    # Remove thinking blocks if present
    try:
      website = re.sub(r"<think>.*?</think>", "", website, flags=re.DOTALL)
    except Exception:
      pass


    # REMOVE MARKDOWN BLOCKS
    website = website.replace("```html", "")
    website = website.replace("```css", "")
    website = website.replace("```js", "")
    website = website.replace("```", "")

    # Extract HTML, CSS, JS safely
    try:
        html_match = re.search(r"--html--(.*?)(--css--|### CSS|$)", website, re.DOTALL)
        css_match = re.search(r"(--css--|### CSS)(.*?)(--js--|### JS|$)", website, re.DOTALL)
        js_match = re.search(r"(--js--|### JS)(.*)", website, re.DOTALL)

        html = html_match.group(1).strip() if html_match else ""
        css = css_match.group(2).strip() if css_match else ""
        js = js_match.group(2).strip() if js_match else ""

    except Exception:
        print("Error parsing website output")
        print(website)
        return


    # Save files
    with open("index.html", "w") as f:
        f.write(html.strip())

    with open("style.css", "w") as f:
        f.write(css.strip())

    with open("script.js", "w") as f:
        f.write(js.strip())


    # Zip files
    with zipfile.ZipFile("website.zip", "w") as zipf:
        zipf.write("index.html")
        zipf.write("style.css")
        zipf.write("script.js")

    print("Website Generated")

## Output Generation (PDF & Website)

In [ ]:
url = "https://www.youtube.com/watch?v=knUMzwkn-fQ"

article = get_smart_summarizer(url)

print(article)

Using LangChain Loader
LangChain failed
Using YouTubeTranscriptApi
Transcript API failed
Using yt-dlp
Transcript found
Using base summarizer
Using Gemini
Gemini failed → Using Groq
**Title:**  
Perfect Chicken Biryani Recipe with Basmati Rice: A Step-by-Step Guide  

**Introduction:**  
This article provides a detailed, recipe-style guide for preparing an authentic chicken biryani using basmati rice. The instructions include precise measurements, marination steps, and cooking times to ensure a flavorful and aromatic dish.  

**Main Points:**  
1. **Ingredients for Basmati Rice Preparation:**  
   - **Basmati Rice:** 3 glasses (750gms), washed thoroughly 2-3 times and soaked for 20 minutes.  
   - **Spices for Sautéing (Shahjeera Layer):**  
     - Shahjeera (poppy seeds): 1 tsp  
     - Chopped mint leaves: 1 cup  
     - Chopped coriander leaves: ½ cup  
     - Chopped green chilies: 6  
     - Cinnamon sticks: 2 inches  
     - Cloves: 4  
     - Cardamom pods: 4  
     - Salt (rock 

In [ ]:
generate_pdf(article)

PDF Generated


In [ ]:
generate_website(article)

Using Gemini
Gemini failed → Using Groq
Website Generated
